# PRUEBA DE ANÁLISIS DE PARADAS

## ANÁLISIS INICIAL
De primeras el análisis se hará sobre una parada concreta, la 5939 correspondiente a la linea 401 siendo la quinta parada de la línea. He escogido esta debido a que la considero muy básica para empezar, puesto que esa parada no tiene correspondencia con ninguna otra línea, por lo tanto los datos de ocupación o de subidas/bajadas no se verían afectados en tanto qué otros autobuses puedan pasar por esa parada, lo cual como caso base es mejor.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math
np.random.seed(495794)


#ABRIR EL CSV COMO PANDAS DATAFRAME
data_input = pd.read_csv("../data/clean_data_5939_new.csv", encoding="utf-8", header=0)
dataset = pd.DataFrame(data_input)

Borramos las columnas que se duplican, como EventTime, TimeSlot e IdExternal, ya que aparecen en ambas tablas mbot y tbso

In [ ]:
dataset.drop(columns=["EventDate.1", "EventTime.1", "TimeSlot.1", "IdExternal.1", "TripId.1"]);

Se eliminan aquellas filas cuya ocupación sea < 0, puesto que como se dijo en el documento adjunto con la información de la base de datos, estas instancias no tienen sentido

In [ ]:
dataset.shape

In [ ]:
dataset = dataset[dataset["Occupation"] >=0]
dataset = dataset[dataset["Occupation"] < 80] #Fijamos 80 como valor arbitrariamente grande, puesto que se ha dado el caso que a veces hay valores extremadamente grandes

In [ ]:
dataset.shape

Se han visto afectadas +- unas 80 filas, teniendo en cuenta que días únicos a día de ejecutar la consulta tenemos unos 120 días, es una media estimada de 2 mediciones erroneas cada 3 días de medición

## ESTUDIO DE LA OCUPACIÓN
### ANÁLISIS DE UN FILTRO BASICO

El objetivo de esta parte es, en base a ciertos filtros, encontrar como resultado variables aleatorias gaussianas. De primeras interpreté o pensé, que un filtro básico sería seleccionar todas las variables de un día, pero el problema de este caso es que las variables conforman una serie temporal a lo largo del día, lo cualhace que las dependencias que existen entre un valor y los antecedentes pasados sea lo suficientemente considerable como para alterar el comportamiento de los datos.

En un caso inicial vamos a tomar el valor de medidas de todoss los lunes de paso de autobús por parada alrededor de las 9 de la mañana, por probar a ver el comportamiento. Para empezar, de cada día únicamente vamos a coger un dato, el más cercano a as 9 de la mañana. Para ello, como la consulta realizada ya está ordenada según franja horaria, bastará con coger el primer dato que aparezca de cada día

In [ ]:
def extract_first_value(data: pd.DataFrame):
    visited_days = []
    return_data = pd.DataFrame(columns=data.columns)
    for index, row in data.iterrows():
        if row["EventDate"] not in visited_days:
            visited_days.append(row["EventDate"])
            return_data.loc[len(return_data)] = row
    return return_data



data_mondays_9am = dataset[dataset["Weekday"] == "L"]
data_mondays_9am = data_mondays_9am[data_mondays_9am["DayType"] == "LA"]
data_mondays_9am = data_mondays_9am[data_mondays_9am["TimeSlot"] == 9]

data_mondays_9am_first = extract_first_value(data_mondays_9am)
data_mondays_9am_first.drop(columns=["EventDate.1", "EventTime.1", "TimeSlot.1", "IdExternal.1", "TripId.1"])

A partir de esto vamos a realizar evaluación de la normalidad de los atributos. Para ello vamos primero a mostrar un histograma de cómo se comporta la distribución de probabilidad

In [ ]:
from scipy.stats import norm
import matplotlib.dates as mdates
#Esto se usará más adelante si se tratan las series temporales a lo largo de cada día

def project_occupation_in_daytime(data_to_project, highlight_trip_id=None):
    data_to_project = data_to_project.copy()
    data_to_project["EventTime"] = pd.to_datetime(
        data_to_project["EventTime"],
        format="%H:%M:%S.%f"
    )

    data_to_project = data_to_project.sort_values("EventTime")

    plt.figure(figsize=(16, 6))

    # Plotear línea completa conectada
    plt.plot(data_to_project["EventTime"], data_to_project["Occupation"],
             marker='o', color='steelblue', alpha=0.7)

    if highlight_trip_id:
        # Superponer puntos del TripId destacado en otro color
        highlight_data = data_to_project[data_to_project["TripId"] == highlight_trip_id]
        plt.scatter(highlight_data["EventTime"], highlight_data["Occupation"],
                    color='red', s=100, zorder=5, label=f'Trip: {highlight_trip_id}')
        plt.legend()

    ax = plt.gca()
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
    ax.xaxis.set_major_locator(mdates.HourLocator(interval=1))

    ax.set_xlim([
        pd.to_datetime("00:00:00", format="%H:%M:%S"),
        pd.to_datetime("23:59:59", format="%H:%M:%S")
    ])

    plt.xlabel('Hora del día')
    plt.ylabel('Occupation')
    plt.title('Evolución de Occupation durante el día')
    plt.xticks(rotation=45)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def histogram(data_hist, variable):
    plt.hist(data_hist[variable], bins= 15)
    plt.xlabel(variable)
    plt.ylabel('Count')
    plt.title(f"Histograma de {variable}")
    plt.xticks(rotation=29)
    plt.grid(True, alpha=0.3)
    plt.show()


#Funcion para visualizar un histograma con su normal teórica ajustada(sigma1 y sigma2 == 0
def histogram_with_normal(data_to_hist, variable):
    data = np.asarray(data_to_hist[variable])
    mu = np.mean(data)
    sigma = np.std(data, ddof=0)

    # Histograma en densidad
    plt.hist(data,bins = 15, density=True, alpha=0.5, label="Datos")
    x = np.linspace(min(data), max(data), 400)
    y = norm.pdf(x, loc=mu, scale=sigma)
    plt.plot(x, y, 'r-', lw=2, label=f"N({mu:.2f}, {sigma:.2f}²)")

    plt.xlabel(variable)
    plt.ylabel('Densidad')
    plt.title(f"Histograma de {variable} + Normal ajustada")
    plt.xticks(rotation=29)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()


histogram(data_mondays_9am_first, "Occupation")

Aquí se ve como la variable occupation tiene una tendencia que a simple vista no se parece mucho a una campana de Gauss, aunque cierto es que con tan pocos datos es dificil a simple vista analizarlo. Para analizar mejor esta tendencia, vamos a someter a estos datos a unos experimentos que miden la simetría y la kurtosis de una variable aleatoria, con el fín de estimar su "normalidad". Los valores de skewness y kurtosis se miden calculando y estandarizando los momentos centrales de una distribución.

Estos momentos centrales se obtienen a partir de la formulación de los momentos algebráicos, denotados $ \mu_r' $ donde $r$ especifica el orden
$$
\mu_r' = E(x^r) = \sum_k k^r\, p(k)
\qquad \text{o} \qquad
\int_{-\infty}^{\infty} x^r f(x)\, dx
$$

Donde $E$ es la esperanza, $x$ y $k$ son representaciones de la variable aleatoria, tanto en su versión discreta($k$), como en su versión contínua($x$) y $r$ es el orden de magnitud del momento a calcular.


Estos momentos se calculan a partir del origen, mientras que los momentos centrales, **$\mu_r$** se calculan sobre el valor de la media, el cual se sabe como el momento algebráico **$\mu_1'$**. Estos valores **$\mu_r$** se obtienen a partir de:
$$
\mu_r = E\left[(k - E(k)\,)^{r}\right]
$$
De esta fórmula general obtenemos los momentos centrales **$\mu_1$** (Media), **$\mu_2$** (Varianza), **$\mu_3$** (skewness)[sin estandarizar] y **$\mu_4$** (kurtosis)[sin estandarizar]. Se puede obtener para una muestra de $n$ observaciones, podemos definir el cálculo de los momentos algebráicos $m_r'$ y de los momentos centrales $m_r$ como:

$$
m_r' = \frac{1} {n} \sum_{x=0}^{n}x^r
\qquad \text{y} \qquad
m_r = \frac{1} {n} \sum_{x=0}^{n}(x - m_1')^r
$$
Por lo tanto, a partir de ahora, como tratamos con muestras, $m_r$ será el equivalente muestral a $\mu_r$.

Ahora, para estandarizar una fórmula general de skewness y kurtosis, se consideran unas fórmulas genéricas a usar en vez de los momentos centrales 3 y 4, que derivan de estos, pero con la siguiente diferencia.

Se considera skewness $\gamma_1$ y kurtosis $\gamma_2$ como:
$$
\gamma_1 = \frac{m_3} {m_2^{3/2}}
\qquad \text{y} \qquad
\gamma_2 = \frac{m_4} {m_2^{2}} - 3
$$

Donde se denota que los exponentes de los denominadores de ambas fórmulas se usan para eliminar las unidades de los valores $\gamma$, puesto que los momentos centrales $m_r$ presentan grado r, y como se comparan estos con la varianza $m_2$ hay que elevar este exponente a lo necesario para que ambos numerador y denominador estén sujetos a las mismas unidades, y el resultado sea adimensional. También se destaca que de la fórmula de la kurtosis se le resta 3 al valor. Esto es debido a que es un estandar considerar kurtosis como el "sobrante" de kurtosis que tiene la distribución normal. Como la distribución gaussiana presenta kurtosis 3, se resta este valor para considerar este exceso.

Y considerando el error de cada uno de los momentos centrales, como la aproximación mediante el teorema central del límite, se obtienen las fórmulas asintóticas para el SE(Error Estandar) de skewness y Kurtosis como:
$$
SE(\gamma_1) = \sqrt{\frac{6}{n}}
\qquad \text{y} \qquad
SE(\gamma_2) = \sqrt{\frac{24}{n}}
$$

Posteriormente se calcula el Z Score tanto para $\gamma_1$ como para $\gamma_2$ como lo siguiente:

$$
Z(\gamma_1) = \frac{\gamma_1} {SE(\gamma_1)}
\qquad \text{y} \qquad
Z(\gamma_12) = \frac{\gamma_2} {SE(\gamma_2)}
$$

Como posibles resultados, según los valores de skewness y kurtosis podemos encontrar lo siguiente:


Analizando según skewness ($\gamma_1$):
- Valor $0$ → indica simetría **perfecta**
- Valor $> 0$ → la cola derecha es mayor que la izquierda
- Valor $< 0$ → la cola izquierda es mayor que la derecha

Analizando según kurtosis ($\gamma_2$)*(Asumiendo kurtosis en exceso)* :
- Valor $0$ → distribución **mesocúrtica**, similar en pico a la normal
- Valor $> 0$ → distribución **leptocúrtica**, con un pico más alto que la normal
- Valor $< 0$ → distribución **platicúrtica**, más achatada que la normal



Para traspasar todo esto a código, se ha hecho uso de las siguientes funciones

In [ ]:


def central_moment(x, r):
    mean = np.mean(x)
    return sum((xi - mean) ** r for xi in x) / len(x)


def kurtosis(x):
    return (central_moment(x, 4) / central_moment(x, 2) ** 2) -3

def skewness(x):
    return central_moment(x, 3) / central_moment(x, 2)**(3/2)


def general_behaviour(skew_value, kurt_value):
    return_statement = ""
    if skew_value == 0:
        if kurt_value == 0:
            return_statement += "\t-Distribución Normal perfecta"
        else:
            return_statement += "\t-Distribución Normal con simetría perfecta, pero valor de kurtosis que indica:\n"
            return_statement += "\t-Distribución leptocúrtica" if kurt_value > 0 else "\t-Distribución platicúrtica"
    else:
        if kurt_value == 0:
            return_statement += "\t-Distribución Normal con kurtosis perfecta, pero valor de skewness que indica:\n"
            return_statement += "\t-Distribución con cola más grande en el lado derecho" if skew_value > 0 else "\t-Distribución con cola más grande en el lado izquierdo"
    if skew_value > 0:
        if kurt_value > 0:
            return_statement += "\t-Cola derecha más grande que izquierda\n\t-Distribución leptocúrtica(picos más altos de lo normal)"
        else:
            return_statement += "\t-Cola derecha más grande que izquierda\n\t-Distribución platicúrtica(distribución más achatada de lo normal)"
    else:
        if kurt_value > 0:
            return_statement += "\t-Cola izquierda más grande que derecha\n\t-Distribución leptocúrtica(picos más altos de lo normal)"
        else:
            return_statement += "\t-Cola izquierda más grande que derecha\n\t-Distribución platicúrtica(distribución más achatada de lo normal)"
    return return_statement


def is_normal(variables: list):
    if len(variables) < 50: #CASO Z score 1.96
        z_crit = 1.96
    elif len(variables) < 300: #CASO Z score 3.29
        z_crit = 3.29
    else:
        z_crit = "False"

    skew = skewness(variables)
    kurt = kurtosis(variables)


    n = len(variables)
    se_skew = math.sqrt(6.0 / n)
    se_kurt = math.sqrt(24.0 / n)

    mean = np.mean(variables)
    std_dev = math.sqrt(central_moment(variables, 2))
    z_skew = skew / se_skew
    z_kurt = kurt / se_kurt

    if z_crit != "False":
        approx_normal = (abs(z_skew) <= z_crit) and (abs(z_kurt) <= z_crit)
        print(f"-Para {n} valores, con valor skewness: {skew} y kurtosis: {kurt}, \n\n"
          f"-Con valores de SE, SE_skewness: {se_skew}, SE_kurtosis: {se_kurt}, \n\n"
          f"-Se obtienen los siguientes valores de Z; Z_skew: {z_skew} y Z_kurtosis: {z_kurt}, \n\n"
          f"-Se concluye por lo tanto que la hipótesis de que la distribución es normal es: {approx_normal}\n\n"
          f"La distribución presenta una media:  {mean} y una desviación estandar σ: {std_dev}\n\n"
          f"-El comportamiento general de esta distribución es:\n{general_behaviour(skew, kurt)}")

    else:
        approx_normal = (abs(skew) <= 2) and (abs(kurt) + 3 <= 7) #para muestras grandes de Z > 300 no se utilizan los errores, sino valores absolutos  de medida
        print(f"-Para {n} valores, con valor skewness: {skew} y kurtosis: {kurt}, \n\n"
              f"-Con valores de SE, SE_skewness: {se_skew}, SE_kurtosis: {se_kurt}, \n\n"
              f"-Se concluye por lo tanto que la hipótesis de que la distribución es normal es: {approx_normal}\n\n"
              f"La distribución presenta una media:  {mean} y una desviación estandar σ: {std_dev}\n\n"
              f"-El comportamiento general de esta distribución es:\n{general_behaviour(skew, kurt)}")
    return None

In [ ]:
is_normal(data_mondays_9am_first["Occupation"])

En este caso se ve como el experimento ha resultado en una aceptación de la hipótesis de normalidad de los datos. Antes de revisar otros días a la misma hora para analizar el comportamiento, vamos a echarle un vistazo a esos casos más alejados de la media que se obtienen en esta distribución. Para ello, de tener una muestra con un número de población mayor, seguiríamos la regla de capturar todos los datos alejados una distancia de $2\sigma$ de la media de la distribución, y a estudiar qué hay(si hay algo) de especial o distinto en esos días. Como de primeras, al trabajar con muestras más pequeñas, vamos a realizar un experimento para identificar aquellos valores detectados como outliers, mediante el test ESD generalizado.

El test ESD Generalizado(*E*xtreme *S*tudentized *D*eviate de las siglas *ESD*) es un test propuesto por Rosner en 1983, el cual, a diferencia de otros test como el de Grubss, o el de Tietjen-Moore que necesitan saber qué numero de outliers se consideran, este test evalúa mediante hipótesis cuál es el número adecuado de outliers para una distribución, para ello recurre a lo siguiente:

El algoritmo define las siguientes hipótesis:

$H_0$: No hay outliers

$H_a$: Hay hasta $r$ outliers

Luego, se sigue un proceso iterativo, en el cual se computa la siguiente fórmula para cada iteración $i$:

$$
R_i = \frac {\max_i{\left|x_i-\bar{x}\right|}}{s}
$$
Donde $\bar{x}$ denota la media y $s$ es la desviación estándar muestral, que difiere de $m_2$(desviación estándar de la población) en que en su cálculose multiplica por $\frac {1} {n - 1}$ en vez de $\frac {1} {n}$
En cada iteración se encuentra el valor que maximiza esa $R_i$, y se almacena. En la siguiente iteración $i+1$ se recalcula esa $R_{i+1}$ sin tener en cuenta el valor anterior

Esto se repite hasta que se han computado los $r$ datos de la $H_a$

Luego, con un nivel definido de significancia $\alpha$, para cada $R_i$ con $i \in 1,2,....,i$ se calcula un valor $\lambda_i$

$$
\lambda_i = \frac {\left ( n - i \right ) t_{p, n-i-1}} {\sqrt{\left ( n- i -1 + t_{p, n-i-1}^{2} \right ) \left ( n -i +1 \right )}}
$$
Donde $t_{p,v}$ se corresponde con el cuantil p de la distribución t con v grados de libertad. En estas fórmulas, $v$ se corresponde con $n-i-1$ y p con
$$
p = 1 - \frac {\alpha} {2 \left ( n - i + 1 \right )}
$$


De todo esto obtenemos que el número de outliers es determinado según el máximo valor $i$ que cumpla que $R_i > \lambda_i$

En código

In [ ]:
from math import sqrt
from scipy.stats import t as t_student


def calculate_lambda(iter_value, n_elems, significance_level: float = 0.05):
    """

    :param iter_value: the iteration i in which calculate the associate lambda
    :param significance_level:
    :param n_elems:
    :return: The lambda value for that iteration
    """
    calculate_p = 1 - (significance_level / (2*(n_elems - iter_value +1)))

    t_p_v = t_student.ppf(calculate_p, (n_elems - iter_value - 1))

    return ((n_elems - iter_value) * t_p_v)/sqrt((n_elems - iter_value - 1 + t_p_v*t_p_v)* (n_elems - iter_value + 1))



def calc_r_value(element, variables):
    """
    Function to calculate the R value of a data given the sample it belongs
    :param element: The element to calculate the R value of
    :param variables: The complete sample to evaluate mean and sample standard deviation
    :return: The r value for that element
    """


    mean = np.mean(variables)
    s = sqrt(sum((xi - mean) ** 2 for xi in variables) / (len(variables) -1))
    return abs(element - mean)/s




def extract_r_parameters(variables: list, r:int):
    """

    :param variables:
    :param r:
    :return:
    """

    #variables = np.array(variables)
    r_parameters = []
    variables_as_dict = {}

    for iteration in range(r):

        for variable in variables:
            variables_as_dict[variable] = calc_r_value(variable, variables)


        item, max_r = 0, 0
        for key,value in variables_as_dict.items():
            if value > max_r:
                max_r = value
                item = key
        r_parameters.append((item, max_r))
        #print(f"DEBUG voy a eliminar {item} con {max_r}")

        variables.remove(item)
        variables_as_dict = {}

    return r_parameters




def estimate_outliers(variables:list, outliers_hipotesis: int, significance_level: float = 0.05):
    """

    :param variables:
    :param outliers_hipotesis:
    :param significance_level:
    :return:
    """
    r_parameters = extract_r_parameters(variables, outliers_hipotesis)

    #FOR THE FORMATTED EXIT
    print(f"H_0:\tThere are No outliers")
    print(f"H_a:\tThere are up to {outliers_hipotesis} outliers in the data")
    print(f"\nSignificance level : {significance_level}")
    print(f"Critical Region: Reject H_0 if R_i > Critical value")
    print(f"\nSummary Table for EDC Test")
    print(f"-----------------------------------------------------------")
    print(f" Exact              Test                        Critical")
    print(f"Number of           Statistic                   Value, λ ")
    print(f"Outliers, i         Value, R_i                     5%")
    print(f"-----------------------------------------------------------")

    for iteration in range(outliers_hipotesis):
        iteration_value = calculate_lambda(iteration, len(variables))

        print(f"\t{iteration + 1}\t\t\t{r_parameters[iteration][1]}\t\t\t{iteration_value}") if iteration_value >= r_parameters[iteration][1] else print(f"\t{iteration + 1}\t\t\t{r_parameters[iteration][1]}\t\t\t{iteration_value} *")

    return r_parameters

outliers_mondays = estimate_outliers(data_mondays_9am_first["Occupation"].tolist(), 4);

Tras la ejecución de este experimento para este dataset, se muestra como no se ha aceptado en ninguno de los casos la Hipótesis alternativa de que existian outliers. No obstante, este experimento en este caso puede estar influido por el poco número muestral que se tiene. Se dejará para experimentos con mayores muestras. Aún así, viendo el histograma:

In [ ]:
histogram(data_mondays_9am_first, "Occupation")

Se observaba como la ocupación en la zona máxima (>30) era "notoria" (conteo de 3 para una muestra de 19), por lo que vamos a ver si esos días tenían algo en común distinto al resto.

In [ ]:
potencial_abnormal_data_mondays = data_mondays_9am_first[data_mondays_9am_first["Occupation"] >= 30]
potencial_abnormal_data_mondays

Algo curioso de primeras que se observa en estos valores es, por ejemplo, que 2 de los Lunes, el 1 de Septiembre y el 8 de Septiembre, los coincidentes con días de mayor ocupación a las 9 de la mañana, son días en los que hubo gratuidad,(Ya que FreeDay = 1 en esas instancias). También cabe destacar que la predicción de temperatura a esa hora de la mañana del día que no presenta gratuidad es 4 y 8~9 grados menor a lo registrado en los otros días. Puesto que este día no tuvo gratuidad, la baja temperatura pudo ser motivo de la alta ocupación.

Con los nuevos  datos surge un nuevo valor, el correspondiente con el 24 de noviembre. Para estudiar mejor estos "outliers" recurriremos luego a otro análisis.



Vamos a revisar más conjuntos de días a ver si el comportamiento es parecido, por ejemplo de los martes.

In [ ]:
data_tuesdays_9am = dataset[dataset["Weekday"] == "M"]

data_tuesdays_9am = data_tuesdays_9am[data_tuesdays_9am["DayType"] == "LA"]

data_tuesdays_9am = data_tuesdays_9am[data_tuesdays_9am["TimeSlot"] == 9]

data_tuesdays_9am_first = extract_first_value(data_tuesdays_9am)

In [ ]:
histogram(data_tuesdays_9am_first, "Occupation")

In [ ]:
is_normal(data_tuesdays_9am_first["Occupation"])

In [ ]:
outliers_tuesdays = estimate_outliers(data_tuesdays_9am_first["Occupation"].tolist(), 5, 0.05)

In [ ]:
outliers_tuesdays[0:3]

Tras ver los datos de los martes a las 9 de la mañana podemos extraer las siguientes conclusiones:
- La distribución **También** es normal, aunque presenta cola derecha mayr que izquierda, y picos más altos de lo normal
- Existe 1 outlier, que se corresponde con el valor 46
- La media entre las distribuciones de los lunes y de los martes es similar (17.42 el lunes VS 19.66 el martes) y su desviación estandar es igual en ambos casos (9.44)
Vamos a mirar más a fondo el valor anómalo para ver que podemos sacar de él.

In [ ]:
outlier_tuesday = data_tuesdays_9am_first[data_tuesdays_9am_first["Occupation"] >= 46]
outlier_tuesday

Se ve como la fecha corresponde con el 23 de septiembre. Concretamente el día consecutivo al lunes 22, día en el cual también hay una presencia anómala de ocupación y no era explicable por gratuidad, por ejemplo.
Vamos a revisar el resto de los dias a ver si esa semana fue problemática también para los miércoles

In [ ]:
data_wednesdays_9am = dataset[dataset["Weekday"] == "X"]

data_wednesdays_9am = data_wednesdays_9am[data_wednesdays_9am["DayType"] == "LA"]


data_wednesdays_9am = data_wednesdays_9am[data_wednesdays_9am["TimeSlot"] == 9]


data_wednesdays_9am_first= extract_first_value(data_wednesdays_9am)

In [ ]:
histogram(data_wednesdays_9am_first, "Occupation")

In [ ]:
is_normal(data_wednesdays_9am_first["Occupation"])

In [ ]:
possible_outliers_wednesday = estimate_outliers(data_wednesdays_9am_first["Occupation"].tolist(), 5, 0.05);

De los miércoles, se puede sacar lo siguiente:
- La distribución también puede asumir la hipótesis de normalidad.
- En las 3 se presenta una cola más alargada en el lado derecho.
- La media presenta un valor algo mayor que la de lunes y martes, aunque tampoco mucho mayor. La desviación estandar también es levemente mayor
- No se reconoce según el ESD ningún outlier

Vamos a ver aún así si alguno de los casos de mayor valor se corresponde con el 24 de septiembre, el cual alargaría la nube que sobrevuela esa semana, debido a que los 2 días anteriores también presentaban un alto valor

In [ ]:
big_data_wednesdays = data_wednesdays_9am_first[data_wednesdays_9am_first["Occupation"] >= 38]

big_data_wednesdays

Otra vez la semana del 22,23 y 24 de septiembre muestra un valor que podría considerarse "fuera de lo común",aunque ahora también aparece otro valor, el día 10 de septiembre, el cual no parece tener una clara motivación por la cual pueda ir más lleno. Una hipótesis podría ser que su timeslot de la hora de salida es 9, pero el de su hora de llegada a la parada es 10, lo que quizas al ser un bus siguiente tenga unos datos distintos.

Por tener todos los días. Vamos a ver también las distribuciones de los jueves y los viernes a ver que conclusiones generales podemos sacar.

In [ ]:
data_thursdays = dataset[dataset["Weekday"] == "J"]

data_thursdays = data_thursdays[data_thursdays["DayType"] == "LA"]

data_thursdays_9am = data_thursdays[data_thursdays["TimeSlot"] == 9]

data_thursdays_9am_first = extract_first_value(data_thursdays_9am)

data_thursdays_9am_first

In [ ]:
histogram(data_thursdays_9am_first, "Occupation")

In [ ]:
is_normal(data_thursdays_9am_first["Occupation"])

In [ ]:
outliers_thursday = estimate_outliers(data_thursdays_9am_first["Occupation"].tolist(), 5, 0.05);

In [ ]:
data_fridays = dataset[dataset["Weekday"] == "V"]

data_fridays = data_fridays[data_fridays["DayType"] == "LA"]

data_fridays_9am = data_fridays[data_fridays["TimeSlot"] == 9]

data_fridays_9am_first = extract_first_value(data_fridays_9am)

data_fridays_9am_first

In [ ]:
histogram(data_fridays_9am_first, "Occupation")

In [ ]:
is_normal(data_fridays_9am_first["Occupation"])

In [ ]:
outiers_fridays = estimate_outliers(data_fridays_9am_first["Occupation"].tolist(), 5, 0.05);

Concluidas todas las ejecuciones podemos ver ciertas cosas:
- Todos los días, la hipótesis de normalidad es asumida como cierta.
- La media de las distribuciones de los días oscila entre 16~17 y 24  y la desviación típica entre 6 y 13.
- Hay un día, en concreto el jueves, que presenta 3 valores que, aunque no son reconocidos como outliers, conviene analizar, ya que son 3 díaas con valor de ocupación >45, lo cual puede ser indicio de un caso especial, o al menos de interés

Vamos a ver esos datos de los jueves en concreto.

In [ ]:
outliers_thursday

In [ ]:
data_outliers_thursday = data_thursdays_9am_first[data_thursdays_9am_first["Occupation"] >= 46]
data_outliers_thursday

No parece haber un factor que correlacione estos días concretos, ya que puedes agrupar 2 de ellos por temperatura similar y el otro por probabilidad de lluvia mayor a 30.
Hay que tener algo en cuenta de este análisis, y es: Estamos seleccionando la instancia de cada día en concreto que más cerca tiene su valor EventTime(El de la tabla Master) de las 9 am, ya que está la query ordenada por Mbot.EventTime. Esto nos puede dejar en que estos días en concreto el bus anterior acumulara muy poca demanda y este bus quede con un número mayor al "normal". Esto entra dentro de las posibilidades.

Ahora bien, podemos, obviando estos casos que de momento no son explicables, asumir que los días laborables en la franja de las 9 de la mañana siguen la misma distribución de probabilidad, y esta puede ser una normal? Vamos a comprobarlo. Para ello, seleccionaremos todos los "primeros" datos que tenemos de cada día y realizaremos los mismos tests.

In [ ]:
data_9_am = dataset[dataset["DayType"] == "LA"]
data_9_am = data_9_am[data_9_am["TimeSlot"] == 9]
data_9_am_first = extract_first_value(data_9_am)

In [ ]:
histogram(data_9_am_first, "Occupation")

In [ ]:
is_normal(data_9_am_first["Occupation"])

In [ ]:
outliers_weekday = estimate_outliers(data_9_am_first["Occupation"].tolist(), 10, 0.05);

In [ ]:
outliers_weekday[0]

Hecho este análisis podemos ver lo siguiente.
Se **RECHAZA** la hipótesis de normalidad de los datos, al superar la Z_skew el valor de Z_score que se utiliza como baremo. El test de outliers indica(aunque el test en principio aplica a datos aparentemente "normales") que el número estimado de outliers es de 1, siendo este el valor máximo de la distribución.

Esto nos da varias preguntas. Es este "dato atípico" que hemos visto antes, aparentemente sin ningún parámetro asociado a ese día que sea fuera de lo normal, el que nos está fastidiando la normalidad?

In [ ]:
data_9_am_first = data_9_am_first[data_9_am_first["Occupation"] <= 58]

is_normal(data_9_am_first["Occupation"])

Si eliminamos ese valor, obtenemos que la hipótesis de la normalidad se asume, con media ~~20 y desviación estandar 10.04, aunque también, gracias al tipo de test que estamos realizando, tenemos más información sobre los datos. Entre ellos, la cola derecha es más grande de la izquierda, y se presentan picos más altos que de normal, lo cual siempre es bueno saberlo

Si por ejemplo ahora eliminamos el criterio de "first" que estabamos cogiendo para cada día, es decir, ya no vamos a tener en cuenta únicamente el dato más cercano a las 9 por día, si no todos los datos cuyo timeSlot sea las 9. La distribución resultante será también normal?? Tendrá valores de media y desviación típica similares??? Vamos a verlo

In [ ]:
histogram(data_9_am, "Occupation")

In [ ]:
is_normal(data_9_am["Occupation"])

In [ ]:
outliers_weekday_no_first_restr = estimate_outliers(data_9_am["Occupation"].tolist(), 15, 0.05);

In [ ]:
outliers_weekday_no_first_restr[0:4]

Sobre la no restriccion de un dato por día nos encontramos que la normalidad también es aceptada como hipótesis, con valores de media y desviación estandar muy similares a lo anterior. También los valores de skewness y kurtosis son igualmente interpretables.

El test de outliers lanza como outliers los valores más altos, los cuales son 69,63,59 y 58, lo cual cuadra con la anterior distribución(Que de hecho tuvimos que quitarle el valor de 59 porque si no la hipótesis de normalidad no se cumplía).



### Siguientes Avances

Ahora bien. Una vez concluido este análisis de la franja horaria para los días laborables nos quedan las siguientes preguntas o casos particualres por hacernos:

- Los valores extremos que hemos notado para ciertos días los cuales no podían tener una explicación como pudiera ser dias con gratuidad, valores de temperatura muy extremos, etc. Existe un motivo por el cuál se produce ese valor tan elevado de ocupación o es simplemente la casuística del problema.
- Este análisis es replicable a festivos, domingos o días no lectivos?
- Notaremos diferencias si analizamos en profundidad días con gratuidad? Hay suficientes para tener una muestra significativa?
- Lo mismo que la anterior pero aplicable a otros filtros.

Se ha actualizado el notebook en base a los nuevos datos entrantes. Vamos a reevaluar los potenciales "outliers" de cada día a ver si se pueden explicar en base a la reduccion de la dotacion esos dias.

In [ ]:
from obtain_allocation import Allocation, format_string_list

potencial_abnormal_data_mondays

Vamos a ver para cada uno de los días qué dotación tenían para, tanto la franja horaria de las 8 como de las 9.

In [ ]:
data_alloc = Allocation(dataset, "2025-07-01")
for idx, row in potencial_abnormal_data_mondays.iterrows():
    data_alloc.day = row.EventDate
    print(f"Para {data_alloc.day} El numero de autobuses que salen de cabecera en Timeslot 8 es: {data_alloc.allocation_per_timeslot[8]} y a las 9 es {data_alloc.allocation_per_timeslot[9]}") if row.FreeDay == 0 else print("")

Si comparamos con cualquier otro dia que entra dentro de la normalidad.

In [ ]:
data_alloc.day = "2025-09-29"
print(f"Para {data_alloc.day} El numero de autobuses que salen de cabecera en Timeslot 8 es: {data_alloc.allocation_per_timeslot[8]} y a las 9 es {data_alloc.allocation_per_timeslot[9]}")

Se puede ver como no hay diferencias significativas entre un día normal que no es outlier con uno que si lo es.

Hemos obviado aquellos dias que habia gratuidad, y la dotación para las 9 es igual en ambos y para las 8 es en un caso 7 y en otro 5. SI miramos a qué autobús corresponde cada valor:

In [ ]:
for idx, row in potencial_abnormal_data_mondays.iterrows():
    data_alloc.day = row.EventDate
    all_data = dataset[dataset["EventDate"] == data_alloc.day]
    project_occupation_in_daytime(all_data, row.TripId) if row.FreeDay == 0 else print("")

Se puede ver como en el caso del 24 de noviembre(el segundo), el valor "outlier" para esa hora concreta, se puede "explicar" por la naturaleza de lo que más o menos se puede observar en la gráfica. Esto es, la alternancia de valores de ocupación muy bajos y muy altos de forma seguida, lo que da a entender, o más o menos se puede ver, que pasaron 2 buses muy seguidos, el primero se llenó, el segundo iba vacío, y el siguiente(El que estabamos estudiando) al tardar más en llegar generó una acumulación de pasajeros.

En cuanto al 22 de septiembre, no se acaba de explicar, pasan muchos buses en el tramo anterior, y a esa hora justo antes de la medicion hay otro bus con una ocupacion similar.

Vamos a ver el martes.

In [ ]:
for idx, row in outlier_tuesday.iterrows():
    data_alloc.day = row.EventDate
    print(f"Para {data_alloc.day} El numero de autobuses que salen de cabecera en Timeslot 8 es: {data_alloc.allocation_per_timeslot[8]} y a las 9 es {data_alloc.allocation_per_timeslot[9]}") if row.FreeDay == 0 else print("")

In [ ]:
for idx, row in outlier_tuesday.iterrows():
    data_alloc.day = row.EventDate
    all_data = dataset[dataset["EventDate"] == data_alloc.day]
    project_occupation_in_daytime(all_data, row.TripId) if row.FreeDay == 0 else print("")

En los 2 primeros casos, el valor maximo siempre se da después de valores de menor ocupación. En el primer caso, pese a que hay mucha oferta en el timeslot anterior, esta se encuentra muy colapsada en el tiempo, y no tan cercana a las 9, lo que puede explicar el aumento de la ocupacion. En cuanto al segundo caso y al tercero, la bajada en la oferta puede ser motivo de este valor más bajo.

Miercoles:

In [ ]:
for idx, row in big_data_wednesdays.iterrows():
    data_alloc.day = row.EventDate
    print(f"Para {data_alloc.day} El numero de autobuses que salen de cabecera en Timeslot 8 es: {data_alloc.allocation_per_timeslot[8]} y a las 9 es {data_alloc.allocation_per_timeslot[9]}") if row.FreeDay == 0 else print("")

In [ ]:
for idx, row in big_data_wednesdays.iterrows():
    data_alloc.day = row.EventDate
    all_data = dataset[dataset["EventDate"] == data_alloc.day]
    project_occupation_in_daytime(all_data, row.TripId) if row.FreeDay == 0 else print("")

El tercer valor se puede explicar debido a que es el único autobus de esa franja horaria, y hay mucha diferencia en hora entre él y su anterior. En los otros 2 vemos comportamiento muy similar, viniendo de un valor muy bajo de ocupacion muy bajo.

Jueves

In [ ]:
for idx, row in data_outliers_thursday.iterrows():
    data_alloc.day = row.EventDate
    print(f"Para {data_alloc.day} El numero de autobuses que salen de cabecera en Timeslot 8 es: {data_alloc.allocation_per_timeslot[8]} y a las 9 es {data_alloc.allocation_per_timeslot[9]}") if row.FreeDay == 0 else print("")

In [ ]:
for idx, row in data_outliers_thursday.iterrows():
    data_alloc.day = row.EventDate
    all_data = dataset[dataset["EventDate"] == data_alloc.day]
    project_occupation_in_daytime(all_data, row.TripId) if row.FreeDay == 0 else print("")

## Ampliacion a todas franjas horarias

Ahora ampliamos todo el tema de analisis a todas las franjas. Para ello usamos la funcion generada en generate_gaussian.py.

In [ ]:
from generate_gaussian import generate_gaussian_per_slot_daily, is_normal_modified


new_dataframe = generate_gaussian_per_slot_daily(dataset, "Occupation")

new_dataframe

In [ ]:
result, data_abnormal = is_normal_modified(dataset, "M", 8, "Occupation")

In [ ]:
histogram_with_normal(pd.DataFrame(data_abnormal), "Occupation")

In [ ]:
result, data_abnormal = is_normal_modified(dataset, "X", 8, "Occupation")

In [ ]:
histogram_with_normal(pd.DataFrame(data_abnormal), "Occupation")

In [ ]:
result, data_abnormal = is_normal_modified(dataset, "J", 8, "Occupation")

In [ ]:
histogram_with_normal(pd.DataFrame(data_abnormal), "Occupation")

In [ ]:
result, data_abnormal = is_normal_modified(dataset, "V", 8, "Occupation")

In [ ]:
histogram_with_normal(pd.DataFrame(data_abnormal), "Occupation")

In [ ]:
dataset_prueba = dataset[dataset["Weekday"] == "V"]
dataset_prueba = dataset_prueba[dataset_prueba["DayType"] == "LA"]
dataset_prueba = dataset_prueba[dataset_prueba["FreeDay"] == 0]
dataset_prueba = dataset_prueba[dataset_prueba["TimeSlot"] == 8]
dataset_prueba = dataset_prueba[dataset_prueba["EventDate"] == "2025-09-26"]
dataset_prueba

## Estudio con subidas y bajadas. No solo occupation

In [ ]:
new_dataframe = generate_gaussian_per_slot_daily(dataset, "Board")

new_dataframe

In [ ]:
new_dataframe = generate_gaussian_per_slot_daily(dataset, "DropOff")

new_dataframe

In [ ]:
result, data_abnormal = is_normal_modified(dataset, "V", 8, "DropOff")

In [ ]:
histogram_with_normal(pd.DataFrame(data_abnormal), "DropOff")